In [0]:
from pyspark.sql import Row
import json
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StringType, MapType
from pyspark.sql import functions as F

# read the bronze layer
bronze_df = spark.read.format("delta").load(
    "abfss://lyon-data@stlyonloyersdev.dfs.core.windows.net/bronze/delta/dvf"
)

In [0]:
silver_df = bronze_df.select(

    # --------------------------------------------
    # Mutation
    # --------------------------------------------
    F.col("idmutation").cast("long").alias("mutation_id"),
    F.col("datemut").cast("date").alias("mutation_date"),
    F.col("anneemut").cast("int").alias("year"),
    F.col("moismut").cast("int").alias("month"),

    # --------------------------------------------
    # Location
    # --------------------------------------------
    F.col("l_codinsee")
        .cast("string")
        .alias("insee_code"),

    # --------------------------------------------
    # Transaction
    # --------------------------------------------
    F.trim(F.col("libnatmut"))
        .alias("mutation_type"),

    F.trim(F.col("libtypbien"))
        .alias("property_type"),

    F.col("vefa")
        .cast("boolean")
        .alias("vefa"),

    F.col("valeurfonc")
        .cast("double")
        .alias("sale_price"),

    # --------------------------------------------
    # Lots / parcels
    # --------------------------------------------
    F.col("nblot")
        .cast("int")
        .alias("nb_lots"),

    F.col("nbpar")
        .cast("int")
        .alias("nb_parcels"),

    # --------------------------------------------
    # Property counts
    # --------------------------------------------
    F.col("nblocapt")
        .cast("int")
        .alias("nb_apartments"),

    F.col("nblocmai")
        .cast("int")
        .alias("nb_houses"),

    # --------------------------------------------
    # Apartments by number of rooms
    # --------------------------------------------
    F.col("nbapt1pp")
        .cast("int")
        .alias("nb_apt_1_room"),

    F.col("nbapt2pp")
        .cast("int")
        .alias("nb_apt_2_rooms"),

    F.col("nbapt3pp")
        .cast("int")
        .alias("nb_apt_3_rooms"),

    F.col("nbapt4pp")
        .cast("int")
        .alias("nb_apt_4_rooms"),

    F.col("nbapt5pp")
        .cast("int")
        .alias("nb_apt_5_rooms"),

    # --------------------------------------------
    # Surface
    # --------------------------------------------
    F.col("sbatapt")
        .cast("double")
        .alias("surface_apartment"),

    F.col("sbatmai")
        .cast("double")
        .alias("surface_house")
)


# ============================================================
# 4. Basic cleaning
# ============================================================

# Empty strings -> NULL
silver_df = silver_df.select([
    F.when(
        F.trim(F.col(c)) == "",
        None
    ).otherwise(F.col(c)).alias(c)
    if dict(silver_df.dtypes)[c] == "string"
    else F.col(c)
    for c in silver_df.columns
])


# ============================================================
# 5. Clean invalid values
# ============================================================

# Sale price cannot be negative
silver_df = silver_df.withColumn(
    "sale_price",
    F.when(
        F.col("sale_price") < 0,
        None
    ).otherwise(F.col("sale_price"))
)


# Number of properties cannot be negative
count_columns = [
    "nb_lots",
    "nb_parcels",
    "nb_apartments",
    "nb_houses",
    "nb_apt_1_room",
    "nb_apt_2_rooms",
    "nb_apt_3_rooms",
    "nb_apt_4_rooms",
    "nb_apt_5_rooms"
]

for column in count_columns:
    silver_df = silver_df.withColumn(
        column,
        F.when(
            F.col(column) < 0,
            None
        ).otherwise(F.col(column))
    )


# Surface cannot be negative
silver_df = silver_df.withColumn(
    "surface_apartment",
    F.when(
        F.col("surface_apartment") < 0,
        None
    ).otherwise(F.col("surface_apartment"))
)

silver_df = silver_df.withColumn(
    "surface_house",
    F.when(
        F.col("surface_house") < 0,
        None
    ).otherwise(F.col("surface_house"))
)


# ============================================================
# 6. Identify Lyon arrondissement
# ============================================================

silver_df = silver_df.withColumn(
    "lyon_arrondissement",

    F.when(F.col("insee_code") == "69381", "Lyon 1er Arrondissement")
     .when(F.col("insee_code") == "69382", "Lyon 2e Arrondissement")
     .when(F.col("insee_code") == "69383", "Lyon 3e Arrondissement")
     .when(F.col("insee_code") == "69384", "Lyon 4e Arrondissement")
     .when(F.col("insee_code") == "69385", "Lyon 5e Arrondissement")
     .when(F.col("insee_code") == "69386", "Lyon 6e Arrondissement")
     .when(F.col("insee_code") == "69387", "Lyon 7e Arrondissement")
     .when(F.col("insee_code") == "69388", "Lyon 8e Arrondissement")
     .when(F.col("insee_code") == "69389", "Lyon 9e Arrondissement")
     .when(F.col("insee_code") == "69266", "Villeurbanne")
     .otherwise(None)
)


# ============================================================
# 7. Remove exact duplicates
# ============================================================

before_count = silver_df.count()

silver_df = silver_df.dropDuplicates()

after_count = silver_df.count()

print(f"Rows before deduplication : {before_count}")
print(f"Rows after deduplication  : {after_count}")
print(f"Duplicates removed        : {before_count - after_count}")




In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("dbx_lyonloyer_dev.silver.dvf")

In [0]:
%sql

SELECT *
FROM dbx_lyonloyer_dev.silver.dvf limit 10;

In [0]:
SILVER_PATH = "abfss://lyon-data@stlyonloyersdev.dfs.core.windows.net/silver/delta/dvf"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_PATH)
)